# Implement Knowledge Distillation - SOLUTION

**Difficulty**: 🟡 Medium

**Companies**: Google, Apple, Meta, Qualcomm, Tesla

---

### Problem Statement

**Knowledge Distillation** is a model compression technique where a small "student" model learns to mimic a large "teacher" model's behavior. Instead of training only on hard labels (one-hot), the student also learns from the teacher's **soft predictions** (probability distributions over classes).

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [ ]:
# Generate synthetic classification data
torch.manual_seed(42)
np.random.seed(42)

n_classes = 6
n_features = 20
n_samples = 5000

X, y = make_classification(
    n_samples=n_samples, n_features=n_features, n_informative=15,
    n_redundant=3, n_classes=n_classes, n_clusters_per_class=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Classes: {n_classes}, Features: {n_features}')
print(f'Class distribution (train): {torch.bincount(y_train).tolist()}')

In [ ]:
# --- Part 1: Distillation Loss ---

def distillation_loss(student_logits, teacher_logits, labels, temperature, alpha):
    """
    Compute the knowledge distillation loss.

    Loss = alpha * T^2 * KL(student_soft || teacher_soft) + (1 - alpha) * CE(student, labels)

    Args:
        student_logits (Tensor): Raw logits from student model, shape (B, C).
        teacher_logits (Tensor): Raw logits from teacher model, shape (B, C). Should be detached.
        labels (Tensor): Ground truth labels, shape (B,).
        temperature (float): Temperature for softening distributions.
        alpha (float): Weight for soft loss vs hard loss.

    Returns:
        loss (Tensor): Combined distillation loss.
    """
    # Soft loss: KL divergence between temperature-scaled distributions
    student_soft = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_soft = F.softmax(teacher_logits / temperature, dim=-1)
    soft_loss = F.kl_div(student_soft, teacher_soft, reduction='batchmean')

    # Hard loss: standard cross-entropy with true labels
    hard_loss = F.cross_entropy(student_logits, labels)

    # Combined loss (T^2 compensates for gradient scaling)
    loss = alpha * (temperature ** 2) * soft_loss + (1 - alpha) * hard_loss
    return loss


# Quick test
test_student = torch.randn(4, n_classes)
test_teacher = torch.randn(4, n_classes)
test_labels = torch.tensor([0, 1, 2, 3])
test_loss = distillation_loss(test_student, test_teacher.detach(), test_labels, temperature=4.0, alpha=0.7)
print(f'Test distillation loss: {test_loss.item():.4f}')

In [ ]:
# --- Part 2: Teacher and Student Models ---

class TeacherModel(nn.Module):
    """Large MLP for classification (the teacher)."""
    def __init__(self, input_dim, n_classes, hidden_dim=256, n_layers=4):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.BatchNorm1d(hidden_dim)]
        for _ in range(n_layers - 1):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.BatchNorm1d(hidden_dim)])
        layers.append(nn.Linear(hidden_dim, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class StudentModel(nn.Module):
    """Small MLP for classification (the student)."""
    def __init__(self, input_dim, n_classes, hidden_dim=64, n_layers=2):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU()]
        for _ in range(n_layers - 1):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.ReLU()])
        layers.append(nn.Linear(hidden_dim, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


teacher = TeacherModel(n_features, n_classes, hidden_dim=256, n_layers=4)
student_distill = StudentModel(n_features, n_classes, hidden_dim=64, n_layers=2)
student_hard = StudentModel(n_features, n_classes, hidden_dim=64, n_layers=2)

print(f'Teacher parameters: {sum(p.numel() for p in teacher.parameters()):,}')
print(f'Student parameters: {sum(p.numel() for p in student_distill.parameters()):,}')
print(f'Compression ratio: {sum(p.numel() for p in teacher.parameters()) / sum(p.numel() for p in student_distill.parameters()):.1f}x')

In [ ]:
# --- Part 3: Train Teacher ---

def train_with_hard_labels(model, X_train, y_train, n_epochs=100, batch_size=128, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    model.train()
    for epoch in range(n_epochs):
        perm = torch.randperm(len(X_train))
        epoch_loss = 0
        n_batches = 0

        for i in range(0, len(X_train), batch_size):
            x = X_train[perm[i:i+batch_size]]
            y = y_train[perm[i:i+batch_size]]

            logits = model(x)
            loss = F.cross_entropy(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        losses.append(epoch_loss / n_batches)
        if (epoch + 1) % 25 == 0:
            print(f'  Epoch {epoch+1}/{n_epochs}, Loss: {losses[-1]:.4f}')

    return losses


print('Training Teacher...')
teacher_losses = train_with_hard_labels(teacher, X_train, y_train, n_epochs=100)

In [ ]:
# --- Part 4: Train Student with Hard Labels Only ---

print('Training Student (hard labels only)...')
student_hard_losses = train_with_hard_labels(student_hard, X_train, y_train, n_epochs=100)

In [ ]:
# --- Part 5: Train Student with Knowledge Distillation ---

def train_with_distillation(student, teacher, X_train, y_train,
                            temperature=4.0, alpha=0.7, n_epochs=100,
                            batch_size=128, lr=1e-3):
    optimizer = torch.optim.Adam(student.parameters(), lr=lr)
    losses = []

    teacher.eval()
    student.train()

    for epoch in range(n_epochs):
        perm = torch.randperm(len(X_train))
        epoch_loss = 0
        n_batches = 0

        for i in range(0, len(X_train), batch_size):
            x = X_train[perm[i:i+batch_size]]
            y = y_train[perm[i:i+batch_size]]

            with torch.no_grad():
                teacher_logits = teacher(x)

            student_logits = student(x)
            loss = distillation_loss(student_logits, teacher_logits, y, temperature, alpha)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        losses.append(epoch_loss / n_batches)
        if (epoch + 1) % 25 == 0:
            print(f'  Epoch {epoch+1}/{n_epochs}, Loss: {losses[-1]:.4f}')

    return losses


print('Training Student (with distillation)...')
student_distill_losses = train_with_distillation(
    student_distill, teacher, X_train, y_train,
    temperature=4.0, alpha=0.7, n_epochs=100
)

In [ ]:
# --- Validation ---

def evaluate(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        logits = model(X_test)
        preds = logits.argmax(dim=-1)
        accuracy = (preds == y_test).float().mean().item()
    return accuracy


teacher_acc = evaluate(teacher, X_test, y_test)
student_hard_acc = evaluate(student_hard, X_test, y_test)
student_distill_acc = evaluate(student_distill, X_test, y_test)

print('=== Results ===')
print(f'Teacher Accuracy:              {teacher_acc:.4f}')
print(f'Student (hard labels only):    {student_hard_acc:.4f}')
print(f'Student (distilled):           {student_distill_acc:.4f}')
print(f'Distillation improvement:      {student_distill_acc - student_hard_acc:+.4f}')

# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(teacher_losses, label='Teacher (hard labels)', linestyle='--')
plt.plot(student_hard_losses, label='Student (hard labels)')
plt.plot(student_distill_losses, label='Student (distilled)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Comparison')
plt.legend()
plt.show()

# Bar chart of accuracies
fig, ax = plt.subplots(figsize=(8, 5))
models = ['Teacher\n(large)', 'Student\n(hard labels)', 'Student\n(distilled)']
accs = [teacher_acc, student_hard_acc, student_distill_acc]
colors = ['#2ecc71', '#e74c3c', '#3498db']
bars = ax.bar(models, accs, color=colors)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.3f}', ha='center', fontsize=12)
ax.set_ylabel('Test Accuracy')
ax.set_title('Knowledge Distillation Results')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

# Checks
print('\n=== Validation ===')
assert teacher_acc > 0.5, f'Teacher accuracy too low: {teacher_acc:.4f}'
print(f'PASSED: Teacher achieves reasonable accuracy ({teacher_acc:.4f})')

assert student_distill_acc > student_hard_acc - 0.05, (
    f'Distilled student ({student_distill_acc:.4f}) should be comparable or better '
    f'than hard-label student ({student_hard_acc:.4f})'
)
print(f'PASSED: Distilled student ({student_distill_acc:.4f}) performs well vs hard-only ({student_hard_acc:.4f})')

print('\nAll validations passed!')